# Description

In this notebook, I will explore:
- How does pytorch perform matmul.
- How to perform low-bit matmul (int8 and int4)

In [1]:
import torch
import time
import torch, time
from torch.profiler import profile, record_function, ProfilerActivity

# 1. Matmul on pytorch

In [2]:
print("\n=== PyTorch Info ===")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"torch.version.cuda: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")


=== PyTorch Info ===
Torch version: 2.5.1+cu121
CUDA available: True
torch.version.cuda: 12.1
GPU name: NVIDIA RTX A5000


In [3]:
n = 8192

for matrix_type in [torch.float32, torch.float16, torch.int8]:
    A = torch.randn(n, n, device="cuda", dtype=matrix_type)
    B = torch.randn(n, n, device="cuda", dtype=matrix_type)

    # Warm-up (important for GPU benchmarking)
    for _ in range(10):
        torch.matmul(A, B)

    torch.cuda.synchronize()

    # Use CUDA events for precise GPU timing
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    C = torch.matmul(A, B)
    end.record()
    torch.cuda.synchronize()

    elapsed_ms = start.elapsed_time(end)  # milliseconds
    elapsed_s = elapsed_ms / 1000.0
    gflops = (2 * n**3) / (elapsed_s * 1e9)

    print(f"Matrix type: {matrix_type}, Time: {elapsed_s:.4f} s, GFLOPS: {gflops:.2f}")

Matrix type: torch.float32, Time: 0.0639 s, GFLOPS: 17206.30
Matrix type: torch.float16, Time: 0.0121 s, GFLOPS: 90995.07


RuntimeError: "normal_kernel_cuda" not implemented for 'Char'

In [4]:
def bench(n=4096, dtype=torch.float32):
    A = torch.randn(n, n, device="cuda", dtype=dtype)
    B = torch.randn(n, n, device="cuda", dtype=dtype)

    # warmup
    for _ in range(5): C = A @ B
    torch.cuda.synchronize()

    # measure
    start = torch.cuda.Event(True); end = torch.cuda.Event(True)
    start.record(); 
    C = A @ B; 
    end.record()    
    torch.cuda.synchronize()
    
    ms = start.elapsed_time(end)
    flops = 2 * n**3
    tflops = flops / (ms/1e3) / 1e12
    print(f"{dtype}: {tflops:.1f} TFLOPs, {ms:.3f} ms")


def profile_once(n=4096, dtype=torch.float32):
    A = torch.randn(n, n, device="cuda", dtype=dtype)
    B = torch.randn(n, n, device="cuda", dtype=dtype)
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                 record_shapes=True, with_stack=False, profile_memory=False) as prof:
        with record_function("matmul"):
            C = A @ B
        torch.cuda.synchronize()
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

profile_once(n=4096, dtype=torch.bfloat16)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 matmul         0.63%     430.598us        98.75%      67.415ms      67.415ms       0.000us         0.00%       1.327ms       1.327ms             1  
                                           aten::matmul         0.04%      24.044us        98.11%      66.984ms      66.984ms       0.000us         0.00%       1.327ms       1.327ms             1  
         

# 2. Matmul with bitsandbytes

In [5]:
import bitsandbytes as bnb
from bitsandbytes.functional import int8_linear_matmul

In [6]:
print("\n=== bitsandbytes Info ===")
print(f"bitsandbytes version: {bnb.__version__}")


=== bitsandbytes Info ===
bitsandbytes version: 0.48.1


In [7]:
A = torch.randint(-128, 127, (1024, 1024), dtype=torch.int8, device='cuda')
B = torch.randint(-128, 127, (1024, 1024), dtype=torch.int8, device='cuda')

C = int8_linear_matmul(A, B)
print(C.dtype, C.shape)

torch.int32 torch.Size([1024, 1024])


## 2.1. Compare time between torch matmul and bitsandbytes matmul

In [8]:
n = 8192

n_iter = 10

In [9]:
# Measure time for torch matmul on bfloat16

A = torch.randn(n, n, device="cuda", dtype=torch.bfloat16)
B = torch.randn(n, n, device="cuda", dtype=torch.bfloat16)

# Warm-up
for _ in range(10):
    C = torch.matmul(A, B)
torch.cuda.synchronize()    

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(n_iter):
    C = torch.matmul(A, B)
end.record()
torch.cuda.synchronize()    

elapsed_ms = start.elapsed_time(end)  # milliseconds
avg_time = elapsed_ms / n_iter 
gflops = (2 * n**3) / (avg_time * 1e9)
print(f"torch bfloat16: Time: {avg_time:.4f} ms, GFLOPS: {gflops:.2f}")

torch bfloat16: Time: 12.0527 ms, GFLOPS: 91.22


In [10]:
A = torch.randint(-128, 127, (n, n), dtype=torch.int8, device='cuda')
B = torch.randint(-128, 127, (n, n), dtype=torch.int8, device='cuda')

# warm up
for _ in range(5): 
    C = int8_linear_matmul(A, B)
torch.cuda.synchronize()

# measure time
n_iter = 10
start = torch.cuda.Event(True); end = torch.cuda.Event(True)
start.record()
for _ in range(n_iter):
    C = int8_linear_matmul(A, B)
end.record()    
torch.cuda.synchronize()

ms = start.elapsed_time(end) / n_iter
gflops = (2 * 1024**3) / (ms/1e3) / 1e9
print(f"bitsandbytes int8 matmul: {gflops:.1f} GFLOPS, {ms:.3f} ms")
print(f"C dtype: {C.dtype}")
print(f"type(C): {type(C)}")

bitsandbytes int8 matmul: 342.2 GFLOPS, 6.275 ms
C dtype: torch.int32
type(C): <class 'torch.Tensor'>


# 3. Vector quantization and matrix quantization

In [15]:
def quantize_tensor_asymmetric_int8(mat:torch.Tensor):
    """
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    min_val = mat.min()
    max_val = mat.max()
    
    qmin = -128
    qmax = 127
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = qmin - torch.round(min_val / scale).item()
    
    q_mat = torch.clamp(torch.round(mat / scale) + zero_point, qmin, qmax).to(torch.int8)
    return q_mat, scale, zero_point

def quantize_tensor_symmetric_int8(mat:torch.Tensor):
    """
    Symmetric quantization to int8.
    mat: input float tensor (e.g., torch.float32 or torch.bfloat16)
    """
    max_val = torch.max(torch.abs(mat))
    
    qmin = -128
    qmax = 127
    scale = max_val / qmax
    zero_point = 0  # For symmetric quantization, zero_point is typically 0
    
    q_mat = torch.clamp(torch.round(mat / scale), qmin, qmax).to(torch.int8)
    
    # convert scale to float16 torch scalar to save memory
    scale = torch.tensor(scale, dtype=torch.float16).item()
    return q_mat, scale, zero_point

def dequantize_tensor_int8(q_mat, scale:float, zero_point:int=0):
    """
    De-quantize an int8 tensor back to float using the provided scale and zero_point.
    q_mat: input int8 tensor
    scale: float scaling factor
    zero_point: integer zero point
    """
    if q_mat.dtype == torch.float16 or q_mat.dtype == torch.bfloat16 or q_mat.dtype == torch.float32:
        if zero_point == 0:
            return scale * q_mat
        return scale * (q_mat - zero_point)
    return scale * (q_mat.float() - zero_point)

def quantization_error(original, dequantized):
    """
    Compute the relative error between the original and dequantized tensors using l2 norm.
    """
    return torch.norm(original - dequantized) / torch.norm(original)

In [16]:
d_type = torch.float16
N = 8000

X = torch.randn(N, N, device='cuda', dtype=d_type)
W = torch.randn(N, N, device='cuda', dtype=d_type)

In [17]:
# warm-up
for _ in range(5):
    A = torch.matmul(X, W)
    
# Measure time
start = torch.cuda.Event(True); end = torch.cuda.Event(True)
start.record()  
for _ in range(100):
    A = torch.matmul(X, W)
end.record()    
torch.cuda.synchronize()

ms = start.elapsed_time(end) / 100
gflops = (2 * 1024**3) / (ms/1e3) / 1e9
print(f"torch.float16 matmul: {gflops:.1f} GFLOPS, {ms:.3f} ms")
print(f"A.dtype: {A.dtype}")
print(f"A.shape: {A.shape}")

torch.float16 matmul: 182.3 GFLOPS, 11.777 ms
A.dtype: torch.float16
A.shape: torch.Size([8000, 8000])


In [21]:
Xq, x_scale, zero_point = quantize_tensor_symmetric_int8(X)
W_quant, w_scale, w_zero_point = quantize_tensor_symmetric_int8(W)

# x_scale_vec = torch.full((N,), x_scale, device='cuda', dtype=torch.float32)
# w_scale_vec = torch.full((N,), w_scale, device='cuda', dtype=torch.float32)
x_scale = torch.tensor(x_scale, dtype=torch.float32, device='cuda')
w_scale = torch.tensor(w_scale, dtype=torch.float32, device='cuda')

# warm-up
for _ in range(5):
    A_q = int8_linear_matmul(Xq, W_quant)
    
# Measure time
start = torch.cuda.Event(True); end = torch.cuda.Event(True)
start.record()  
for _ in range(100):
    A_q = int8_linear_matmul(Xq, W_quant)
    A_deq = bnb.functional.int8_mm_dequant(A_q, x_scale, w_scale)
end.record()    
torch.cuda.synchronize()

ms = start.elapsed_time(end) / 100
gflops = (2 * 1024**3) / (ms/1e3) / 1e9
print(f"bitsandbytes int8 matmul + dequantization: {gflops:.1f} GFLOPS, {ms:.3f} ms")

error = quantization_error(A, A_deq)
print(f"Quantization relative error (l2 norm): {error.item():.6f}")

/tmp/ipykernel_2489220/259139236.py:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  scale = torch.tensor(scale, dtype=torch.float16).item()


bitsandbytes int8 matmul + dequantization: 316.5 GFLOPS, 6.786 ms
Quantization relative error (l2 norm): nan
